# Адаптер правок `pref_edit` под версию промпта коллеги (Colab A100 80 ГБ)

`pref_edit_v1` (пары «правильный / сломанный вызов `edit_file`») с промптом v3 дал на Kaggle 0.20 против 0.15 без него.
Этот ноутбук переучивает тот же адаптер под любую версию промпта: берёт **те же пары** (`pref_edit.jsonl`), заменяет в них
системное сообщение на `prompts/system.md` из отправки коллеги, обучает `train.pref_lora` и собирает готовый
`submission.zip` (отправка коллеги + строка `adapter:` + файлы адаптера).

В Drive (`MyDrive/gemma_adapters/`) должны лежать: `pref_edit.jsonl` (исходные 188 пар, `data/pref_edit.jsonl` из проекта)
и zip отправки коллеги (например `submissions/teammate_v16.zip`).

Порядок: **0** параметры, **1** карта, **2** код, **3** Drive и пары, **4** модель, **5** обучение (~2,5 ч, vLLM не запускать),
**6** сохранение, **7** архив отправки. После перезапуска среды: 0-4 и дальше с нужной ячейки.

## 0. Параметры (правьте только здесь)

In [ ]:
VERSION = "v16"                                  # версия промпта коллеги
SUBMISSION_ZIP = f"submissions/teammate_{VERSION}.zip"   # путь внутри gemma_adapters
PAIRS = "pref_edit.jsonl"                        # исходные пары (188), путь внутри gemma_adapters
ADAPTER = f"pref_edit_{VERSION}"                 # имя нового адаптера
EPOCHS = 2
MAX_LEN = 8192                                   # потолок на A100 80 ГБ

## 1. Карта (нужна A100 80 ГБ, память свободна)

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
print(out)
name, total, used = [x.strip() for x in out.split(",")]
assert int(total) >= 79000, f"нужна карта на 80 ГБ, а здесь {total} МиБ: смените среду выполнения"
assert int(used) < 2000, f"занято {used} МиБ: остановите vLLM (!pkill -f vllm) или перезапустите среду"

## 2. Код и зависимости

In [ ]:
import os
if not os.path.exists("/content/gemma4-swe-rl"):
    !git clone -q https://github.com/aletonfdg/gemma4-swe-rl.git /content/gemma4-swe-rl
%cd /content/gemma4-swe-rl
!git pull -q
!git log --oneline | head -3
# torch и transformers в Colab уже есть; не обновляем их, чтобы не сломать совместимость
!pip install -q compressed-tensors bitsandbytes accelerate safetensors huggingface_hub kagglehub pyyaml

## 3. Drive, отправка коллеги и пары с её промптом
Пары не меняются: подменяется только системное сообщение. Так разница с `pref_edit_v1` -- ровно в промпте.

In [ ]:
import json, shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/gemma_adapters"
for p in (SUBMISSION_ZIP, PAIRS):
    assert os.path.exists(f"{DRIVE}/{p}"), f"нет {DRIVE}/{p}"

SUB_DIR = "/content/submission_src"
shutil.rmtree(SUB_DIR, ignore_errors=True)
zipfile.ZipFile(f"{DRIVE}/{SUBMISSION_ZIP}").extractall(SUB_DIR)
root = next(d for d, _, files in os.walk(SUB_DIR) if "agent.yaml" in files)   # архив бывает с папкой верхнего уровня
if root != SUB_DIR:
    tmp = SUB_DIR + "_tmp"; shutil.move(root, tmp); shutil.rmtree(SUB_DIR); shutil.move(tmp, SUB_DIR)
system = open(f"{SUB_DIR}/prompts/system.md", encoding="utf-8").read()
print(open(f"{SUB_DIR}/agent.yaml").read())
print("промпт:", len(system), "символов")

os.makedirs("data", exist_ok=True)
DATA = f"data/{ADAPTER}.jsonl"
n = 0
with open(f"{DRIVE}/{PAIRS}", encoding="utf-8") as src, open(DATA, "w", encoding="utf-8") as out:
    for line in src:
        if not line.strip():
            continue
        rec = json.loads(line)
        for side in ("chosen", "rejected"):
            assert rec[side][0]["role"] == "system", "в паре нет системного сообщения"
            rec[side][0]["content"] = system
        out.write(json.dumps(rec, ensure_ascii=False) + "\n")
        n += 1
print("пар с промптом", VERSION, ":", n)
assert n >= 150, "ожидалось ~188 пар"

## 4. Базовая модель в `/content/model` (если уже есть, ничего не качает)

In [ ]:
import getpass
MODEL_DIR = "/content/model"
HF_REPO = "google/gemma-4-31b-it-qat-w4a16-ct"
KAGGLE_MODEL = "google/gemma-4/transformers/gemma-4-31b-it-qat-w4a16-ct"
if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    try:
        from huggingface_hub import snapshot_download
        hf_token = getpass.getpass("HF токен (Enter, если не нужен): ").strip() or None
        snapshot_download(HF_REPO, local_dir=MODEL_DIR, token=hf_token)
    except Exception as e:
        print("HF не получилось:", type(e).__name__, str(e)[:300])
        import kagglehub
        os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("KAGGLE_API_TOKEN (KGAT_...): ").strip()
        path = kagglehub.model_download(KAGGLE_MODEL)
        !cp -r "{path}"/. {MODEL_DIR}/
os.environ["GEMMA_MODEL_DIR"] = MODEL_DIR
!du -sh {MODEL_DIR}

## 5. Обучение (~2,5 ч; vLLM не запускать)
В логе: `пар: N` -- сколько влезло в MAX_LEN (промпт длиннее v3, поэтому часть может отпасть; если меньше ~150,
сообщите), `eval до/в конце` -- доля пар chosen>rejected (у `pref_edit_v1` было 0.57 -> 1.00), `pref-loss` вниз от ~0.69.
При `CUDA out of memory` поставьте в ячейке 0 `MAX_LEN = 6144` и повторите 0 и 5.

In [ ]:
LOG = f"/content/{ADAPTER}.log"
!python -m train.pref_lora --data {DATA} --name {ADAPTER} --decompress --epochs {EPOCHS} --max-len {MAX_LEN} 2>&1 | tee {LOG}

## 6. Сохранить адаптер и лог в Drive

In [ ]:
assert os.path.exists(f"adapters/{ADAPTER}/adapter_model.safetensors"), "адаптер не сохранился, смотрите лог"
!cp -r adapters/{ADAPTER} {DRIVE}/ && cp {LOG} {DRIVE}/
!ls -la {DRIVE}/{ADAPTER}
!grep -E "пар:|eval" {LOG}

## 7. Архив отправки: отправка коллеги + `adapter:` + файлы адаптера
Проверки: файлы в корне архива, распакованный размер < 3 ГиБ. Результат: `gemma_adapters/upload_{ADAPTER}/submission.zip`.

In [ ]:
import re
OUT_DIR = f"/content/upload_{ADAPTER}"
shutil.rmtree(OUT_DIR, ignore_errors=True)
shutil.copytree(SUB_DIR, OUT_DIR)
shutil.rmtree(f"{OUT_DIR}/adapters", ignore_errors=True)
os.makedirs(f"{OUT_DIR}/adapters/{ADAPTER}")
for f in ("adapter_config.json", "adapter_model.safetensors"):
    shutil.copy(f"adapters/{ADAPTER}/{f}", f"{OUT_DIR}/adapters/{ADAPTER}/{f}")
agent = open(f"{OUT_DIR}/agent.yaml", encoding="utf-8").read()
agent = re.sub(r"(?m)^adapter:.*\n?", "", agent)
agent = re.sub(r"(?m)^(model:.*\n)", rf"\1adapter: {ADAPTER}\n", agent, count=1)
assert f"adapter: {ADAPTER}" in agent, "в agent.yaml нет строки model:"
open(f"{OUT_DIR}/agent.yaml", "w", encoding="utf-8").write(agent)
print(agent)
size = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(OUT_DIR) for f in fs)
print(f"распакованный размер: {size / 2**30:.3f} ГиБ")
assert size < 3 * 2**30
ZIP = f"/content/submission_{ADAPTER}.zip"
if os.path.exists(ZIP):
    os.remove(ZIP)
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for d, _, fs in os.walk(OUT_DIR):
        for f in fs:
            p = os.path.join(d, f)
            z.write(p, os.path.relpath(p, OUT_DIR))
print(*sorted(zipfile.ZipFile(ZIP).namelist()), sep="\n")
os.makedirs(f"{DRIVE}/upload_{ADAPTER}", exist_ok=True)
shutil.copy(ZIP, f"{DRIVE}/upload_{ADAPTER}/submission.zip")
print("готово:", f"{DRIVE}/upload_{ADAPTER}/submission.zip")